In [ ]:
## SCENARIO 1
## TIME OF INTRODUCTION: JAN 1
#Heatmap for initial mite loads that can cause colony collapse
# t=0 corresponds to Jan 1
# H_crit = 1000           # collapse threshold for hive bees

import numpy as np
import matplotlib.pyplot as plt

from symengine import exp, cos, pi, Abs
from jitcdde import y, t, jitcdde

# Parameters
phi_B     = 0.018
phi_H     = 0.007
epsilon_B = 2000
b         = 500
k         = 5000
a         = 500
tau       = 12
gamma_B   = 0.003
gamma_H   = 0.05
delta_1   = 0.06
delta_2   = 0.06
epsilon_T = 1
gamma_T   = 1/24
chi       = 600
sigma     = 300
theta     = 105          

kappa     = 1/9

# Initial conditions
B0_base = 5000
H0_base = 10000
f0_base = 1000

# SIMULATION SETUP
STARTTIME = 0
STOPTIME  = 365
DT        = 0.02       
eps       = 1e-6

# Collapse definition
H_crit = 1000           # collapse threshold for hive bees

# Euler-like stabilization
lam = 120                

def Phi_numeric(tt):
    return chi + sigma * np.cos(2*np.pi*(tt - theta)/365)

def build_and_run(T0, V0, B0=B0_base, H0=H0_base, f0=f0_base):

    # Compute I0 from constant past assumption 
    q0 = delta_1 * T0/(a + B0) + delta_2 * V0/(a + B0)
    I0 = q0 * tau

    # Helpers
    def pos(x):
        return (x + Abs(x))/2

    # States
    IDX_B = 0
    IDX_H = 1
    IDX_T = 2
    IDX_V = 3
    IDX_f = 4
    IDX_I = 5

    B = y(IDX_B); H = y(IDX_H); T = y(IDX_T); V = y(IDX_V); f = y(IDX_f); I = y(IDX_I)

    B_tau = y(IDX_B, t - tau)
    T_tau = y(IDX_T, t - tau)
    V_tau = y(IDX_V, t - tau)

    B_pos     = pos(B);     H_pos     = pos(H);     T_pos     = pos(T)
    V_pos     = pos(V);     f_pos     = pos(f);     I_pos     = pos(I)
    B_tau_pos = pos(B_tau); T_tau_pos = pos(T_tau); V_tau_pos = pos(V_tau)

    # MODEL TERMS
    S = (f_pos/(b + f_pos)) * (H_pos/(k + H_pos))

    infestation = delta_1 * (B_pos/(a + B_pos)) * T_pos
    infection   = delta_2 * (B_pos/(a + B_pos)) * V_pos

    q_now = delta_1 * T_pos/(a + B_pos) + delta_2 * V_pos/(a + B_pos)
    q_tau = delta_1 * T_tau_pos/(a + B_tau_pos) + delta_2 * V_tau_pos/(a + B_tau_pos)
    dI    = q_now - q_tau

    exp_B = exp(-gamma_B * tau - I_pos)

    Phi = chi + sigma * cos(2*pi*(t - theta)/365)   # Jan 1 reference

    dB = epsilon_B*S - infestation - infection - gamma_B*B_pos - kappa*exp_B*B_tau_pos
    dH = kappa*exp_B*B_tau_pos - gamma_H*H_pos
    dT = epsilon_T*infestation - gamma_T*T_pos
    dV = epsilon_T*infection   - gamma_T*V_pos
    df = Phi*(H_pos/(k + H_pos)) - phi_B*B_pos - phi_H*H_pos

    # Stabilization/projection
    dB += lam*(B_pos - B)
    dH += lam*(H_pos - H)
    dT += lam*(T_pos - T)
    dV += lam*(V_pos - V)
    df += lam*(f_pos - f)
    dI += lam*(I_pos - I)

    DDE = jitcdde([dB, dH, dT, dV, df, dI])

    def history(s):
        return [B0, H0, T0, V0, f0, I0]

    DDE.past_from_function(history)
    DDE.step_on_discontinuities()
    DDE.set_integration_parameters(atol=1e-8, rtol=1e-6, min_step=1e-12, max_step=DT, first_step=DT)

    # Integrate forward
    t0 = float(DDE.t)
    start = max(STARTTIME + eps, t0)
    times = np.arange(start, STOPTIME + DT/2, DT)

    minH = H0
    t_collapse = np.nan

    for tt in times:
        state = DDE.integrate(float(tt))
        Hval = float(state[IDX_H])
        if Hval < minH:
            minH = Hval
        if np.isnan(t_collapse) and (Hval < H_crit):
            t_collapse = float(tt)
            # Possible to break early to speed up:
            break

    return minH, t_collapse

# GRID SEARCH SETTINGS 
T_vals = np.arange(0, 101, 10)   # adjust ranges/steps as needed
V_vals = np.arange(0, 101, 10)

minH_map = np.zeros((len(V_vals), len(T_vals)))
tc_map   = np.full((len(V_vals), len(T_vals)), np.nan)

# Run grid
for iV, V0 in enumerate(V_vals):
    for iT, T0 in enumerate(T_vals):
        minH, tc = build_and_run(T0=float(T0), V0=float(V0))
        minH_map[iV, iT] = minH
        tc_map[iV, iT]   = tc

# Boolean collapse map
collapse_map = np.isfinite(tc_map)  # collapse if time-to-collapse exists

In [ ]:
# Print a few "near-threshold" points (first collapses in each T0 column)
print("Approximate threshold (smallest V0 that collapses) for each T0:")
for iT, T0 in enumerate(T_vals):
    idx = np.where(collapse_map[:, iT])[0]
    if len(idx) == 0:
        print(f"T0={T0:>4}: no collapse up to V0={V_vals[-1]}")
    else:
        V_thr = V_vals[idx[0]]
        print(f"T0={T0:>4}: V0 ~ {V_thr}")

# PLOTS
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

line_color = (230/255, 97/255, 0/255)

fig, ax = plt.subplots(figsize=(5, 4))

# Lighter viridis
base = plt.colormaps["viridis"]
viridis_light = LinearSegmentedColormap.from_list(
    "viridis_light", base(np.linspace(0.22, 1.00, 256))
)

# Build true cell edges over the displayed domain (prevents clipped corner artifacts)
nV, nT = minH_map.shape
T_edges = np.linspace(T_vals[0], T_vals[-1], nT + 1)
V_edges = np.linspace(V_vals[0], V_vals[-1], nV + 1)

# Centers for contour
T_centers = 0.5 * (T_edges[:-1] + T_edges[1:])
V_centers = 0.5 * (V_edges[:-1] + V_edges[1:])
TTc, VVc = np.meshgrid(T_centers, V_centers)

# Heatmap with visible grid borders
im = ax.pcolormesh(
    T_edges, V_edges, minH_map,
    shading="flat",
    cmap=viridis_light,
    edgecolors= (1, 1, 1, 0.75), #(0, 0, 0, 0.55),
    linewidth=0.6,
    antialiased=False,   # key to avoid corner-cut artifacts
    vmin=1000, vmax=10000,
    zorder=1
)

# Contour aligned to cell centers
ax.contour(
    TTc, VVc, minH_map,
    levels=[H_crit],
    colors=[line_color],
    linewidths=2.5,
    zorder=10
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Minimum hive bee population", fontsize=15)
cbar.ax.tick_params(labelsize=18, width=1.6, length=6, colors='black')
cbar.set_ticks([1000, 5000, 10000])
cbar.set_ticklabels(['1,000', '5,000', '10,000'])

ax.set_xlim(T_vals[0], T_vals[-1])
ax.set_ylim(V_vals[0], V_vals[-1])
ax.margins(x=0, y=0)

ax.set_xlabel("Initial number of virus-free mites, $T(0)$", fontsize=15)
ax.set_ylabel("Initial number of \n virus-carrying mites, $V(0)$", fontsize=15)
ax.tick_params(axis='both', labelsize=18, width=1.6, length=6, colors='black')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticks([0, 50, 100])
ax.set_yticks([0, 50, 100])
ax.text(-0.13, 1.15, "(a)", transform=ax.transAxes, color='black', fontsize=20, va='top', ha='left')

plt.savefig("(a)_Scenario 1.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## SCENARIO 1
## TIME OF INTRODUCTION: APR 15
# Heatmap for initial mite loads that can cause colony collapse
# Calendar convention:
#   t = 0   -> Jan 1
#   t = 105 -> Apr 15 (mites introduced here)
# Simulation runs from t=105 to t=470 (= 365 + 105)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from symengine import exp, cos, pi, Abs
from jitcdde import y, t, jitcdde

# Parameters
phi_B     = 0.018
phi_H     = 0.007
epsilon_B = 2000
b         = 500
k         = 5000
a         = 500
tau       = 12
gamma_B   = 0.003
gamma_H   = 0.05
delta_1   = 0.06
delta_2   = 0.06
epsilon_T = 1
gamma_T   = 1/24
chi       = 600
sigma     = 300
theta     = 105
kappa     = 1/9

# Initial colony conditions
B0_base = 5000
H0_base = 10000
f0_base = 1000

# Calendar shift (Apr 15 from Jan 1 in non-leap year)
DAY0_SHIFT = 105

# Simulation setup
STARTTIME = DAY0_SHIFT
STOPTIME  = 365 + DAY0_SHIFT
DT        = 0.02
eps       = 1e-6

# Collapse threshold
H_crit = 1000

# Stabilization
lam = 120

def Phi_numeric(tt):
    # t is true calendar time (Jan 1 = 0), so no extra shift here
    return chi + sigma * np.cos(2 * np.pi * (tt - theta) / 365)

def build_and_run(T0, V0, B0=B0_base, H0=H0_base, f0=f0_base):
    # Constant-past approximation for delay variable at introduction time
    q0 = delta_1 * T0 / (a + B0) + delta_2 * V0 / (a + B0)
    I0 = q0 * tau

    def pos(x):
        return (x + Abs(x)) / 2

    IDX_B, IDX_H, IDX_T, IDX_V, IDX_f, IDX_I = 0, 1, 2, 3, 4, 5

    B, H, Tm, V, f, I = y(IDX_B), y(IDX_H), y(IDX_T), y(IDX_V), y(IDX_f), y(IDX_I)
    B_tau = y(IDX_B, t - tau)
    T_tau = y(IDX_T, t - tau)
    V_tau = y(IDX_V, t - tau)

    B_pos, H_pos, T_pos = pos(B), pos(H), pos(Tm)
    V_pos, f_pos, I_pos = pos(V), pos(f), pos(I)
    B_tau_pos, T_tau_pos, V_tau_pos = pos(B_tau), pos(T_tau), pos(V_tau)

    # Model terms
    S = (f_pos / (b + f_pos)) * (H_pos / (k + H_pos))

    infestation = delta_1 * (B_pos / (a + B_pos)) * T_pos
    infection   = delta_2 * (B_pos / (a + B_pos)) * V_pos

    q_now = delta_1 * T_pos / (a + B_pos) + delta_2 * V_pos / (a + B_pos)
    q_tau = delta_1 * T_tau_pos / (a + B_tau_pos) + delta_2 * V_tau_pos / (a + B_tau_pos)
    dI    = q_now - q_tau

    exp_B = exp(-gamma_B * tau - I_pos)

    # Calendar-based forcing (Jan 1 reference)
    Phi = chi + sigma * cos(2 * pi * (t - theta) / 365)

    dB = epsilon_B * S - infestation - infection - gamma_B * B_pos - kappa * exp_B * B_tau_pos
    dH = kappa * exp_B * B_tau_pos - gamma_H * H_pos
    dT = epsilon_T * infestation - gamma_T * T_pos
    dV = epsilon_T * infection   - gamma_T * V_pos
    df = Phi * (H_pos / (k + H_pos)) - phi_B * B_pos - phi_H * H_pos

    # Stabilization/projection
    dB += lam * (B_pos - B)
    dH += lam * (H_pos - H)
    dT += lam * (T_pos - Tm)
    dV += lam * (V_pos - V)
    df += lam * (f_pos - f)
    dI += lam * (I_pos - I)

    DDE = jitcdde([dB, dH, dT, dV, df, dI])

    def history(_s):
        # Mites introduced at t=105 by starting integration at STARTTIME
        return [B0, H0, T0, V0, f0, I0]

    DDE.past_from_function(history)
    DDE.step_on_discontinuities()
    DDE.set_integration_parameters(
        atol=1e-8, rtol=1e-6, min_step=1e-12, max_step=DT, first_step=DT
    )

    t0 = float(DDE.t)
    start = max(STARTTIME + eps, t0)
    times = np.arange(start, STOPTIME + DT / 2, DT)

    minH = H0
    t_collapse = np.nan

    for tt in times:
        state = DDE.integrate(float(tt))
        Hval = float(state[IDX_H])

        if Hval < minH:
            minH = Hval

        if np.isnan(t_collapse) and (Hval < H_crit):
            t_collapse = float(tt)
            break

    return minH, t_collapse

# Grid search
T_vals = np.arange(0, 101, 10)
V_vals = np.arange(0, 101, 10)

minH_map = np.zeros((len(V_vals), len(T_vals)))
tc_map   = np.full((len(V_vals), len(T_vals)), np.nan)

for iV, V0 in enumerate(V_vals):
    for iT, T0 in enumerate(T_vals):
        minH, tc = build_and_run(T0=float(T0), V0=float(V0))
        minH_map[iV, iT] = minH
        tc_map[iV, iT]   = tc

collapse_map = np.isfinite(tc_map)

In [ ]:
# Print a few "near-threshold" points (first collapses in each T0 column)
print("Approximate threshold (smallest V0 that collapses) for each T0:")
for iT, T0 in enumerate(T_vals):
    idx = np.where(collapse_map[:, iT])[0]
    if len(idx) == 0:
        print(f"T0={T0:>4}: no collapse up to V0={V_vals[-1]}")
    else:
        V_thr = V_vals[idx[0]]
        print(f"T0={T0:>4}: V0 ~ {V_thr}")

# PLOTS
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

line_color = (230/255, 97/255, 0/255)

fig, ax = plt.subplots(figsize=(5, 4))

# Lighter viridis
base = plt.colormaps["viridis"]
viridis_light = LinearSegmentedColormap.from_list(
    "viridis_light", base(np.linspace(0.22, 1.00, 256))
)

# Build true cell edges over the displayed domain (prevents clipped corner artifacts)
nV, nT = minH_map.shape
T_edges = np.linspace(T_vals[0], T_vals[-1], nT + 1)
V_edges = np.linspace(V_vals[0], V_vals[-1], nV + 1)

# Centers for contour
T_centers = 0.5 * (T_edges[:-1] + T_edges[1:])
V_centers = 0.5 * (V_edges[:-1] + V_edges[1:])
TTc, VVc = np.meshgrid(T_centers, V_centers)

# Heatmap with visible grid borders
im = ax.pcolormesh(
    T_edges, V_edges, minH_map,
    shading="flat",
    cmap=viridis_light,
    edgecolors= (1, 1, 1, 0.75), #(0, 0, 0, 0.55),
    linewidth=0.6,
    antialiased=False,   # key to avoid corner-cut artifacts
    vmin=1000, vmax=10000,
    zorder=1
)

# Contour aligned to cell centers
ax.contour(
    TTc, VVc, minH_map,
    levels=[H_crit],
    colors=[line_color],
    linewidths=2.5,
    zorder=10
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Minimum hive bee population", fontsize=15)
cbar.ax.tick_params(labelsize=18, width=1.6, length=6, colors='black')
cbar.set_ticks([1000, 5000, 10000])
cbar.set_ticklabels(['1,000', '5,000', '10,000'])

ax.set_xlim(T_vals[0], T_vals[-1])
ax.set_ylim(V_vals[0], V_vals[-1])
ax.margins(x=0, y=0)

ax.set_xlabel("Initial number of virus-free mites, $T(0)$", fontsize=15)
ax.set_ylabel("Initial number of \n virus-carrying mites, $V(0)$", fontsize=15)
ax.tick_params(axis='both', labelsize=18, width=1.6, length=6, colors='black')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticks([0, 30, 100])
ax.set_yticks([0, 30, 100])
ax.text(-0.13, 1.15, "(b)", transform=ax.transAxes, color='black', fontsize=20, va='top', ha='left')
plt.savefig("(b)_Scenario 1.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## SCENARIO 1
## TIME OF INTRODUCTION: OCT 15
# Heatmap for initial mite loads that can cause colony collapse
# Calendar convention:
#   t = 0   -> Jan 1
#   t = 287 -> Oct 15 (mites introduced here)
# Simulation runs from t=105 to t=470 (= 365 + 105)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from symengine import exp, cos, pi, Abs
from jitcdde import y, t, jitcdde

# Parameters
phi_B     = 0.018
phi_H     = 0.007
epsilon_B = 2000
b         = 500
k         = 5000
a         = 500
tau       = 12
gamma_B   = 0.003
gamma_H   = 0.05
delta_1   = 0.06
delta_2   = 0.06
epsilon_T = 1
gamma_T   = 1/24
chi       = 600
sigma     = 300
theta     = 105
kappa     = 1/9

# Initial colony conditions
B0_base = 5000
H0_base = 10000
f0_base = 1000

# Calendar shift (Oct 15 from Jan 1 in non-leap year)
DAY0_SHIFT = 287

# Simulation setup
STARTTIME = DAY0_SHIFT
STOPTIME  = 365 + DAY0_SHIFT
DT        = 0.02
eps       = 1e-6

# Collapse threshold
H_crit = 1000

# Stabilization
lam = 120

def Phi_numeric(tt):
    # t is true calendar time (Jan 1 = 0), so no extra shift here
    return chi + sigma * np.cos(2 * np.pi * (tt - theta) / 365)

def build_and_run(T0, V0, B0=B0_base, H0=H0_base, f0=f0_base):
    # Constant-past approximation for delay variable at introduction time
    q0 = delta_1 * T0 / (a + B0) + delta_2 * V0 / (a + B0)
    I0 = q0 * tau

    def pos(x):
        return (x + Abs(x)) / 2

    IDX_B, IDX_H, IDX_T, IDX_V, IDX_f, IDX_I = 0, 1, 2, 3, 4, 5

    B, H, Tm, V, f, I = y(IDX_B), y(IDX_H), y(IDX_T), y(IDX_V), y(IDX_f), y(IDX_I)
    B_tau = y(IDX_B, t - tau)
    T_tau = y(IDX_T, t - tau)
    V_tau = y(IDX_V, t - tau)

    B_pos, H_pos, T_pos = pos(B), pos(H), pos(Tm)
    V_pos, f_pos, I_pos = pos(V), pos(f), pos(I)
    B_tau_pos, T_tau_pos, V_tau_pos = pos(B_tau), pos(T_tau), pos(V_tau)

    # Model terms
    S = (f_pos / (b + f_pos)) * (H_pos / (k + H_pos))

    infestation = delta_1 * (B_pos / (a + B_pos)) * T_pos
    infection   = delta_2 * (B_pos / (a + B_pos)) * V_pos

    q_now = delta_1 * T_pos / (a + B_pos) + delta_2 * V_pos / (a + B_pos)
    q_tau = delta_1 * T_tau_pos / (a + B_tau_pos) + delta_2 * V_tau_pos / (a + B_tau_pos)
    dI    = q_now - q_tau

    exp_B = exp(-gamma_B * tau - I_pos)

    # Calendar-based forcing (Jan 1 reference)
    Phi = chi + sigma * cos(2 * pi * (t - theta) / 365)

    dB = epsilon_B * S - infestation - infection - gamma_B * B_pos - kappa * exp_B * B_tau_pos
    dH = kappa * exp_B * B_tau_pos - gamma_H * H_pos
    dT = epsilon_T * infestation - gamma_T * T_pos
    dV = epsilon_T * infection   - gamma_T * V_pos
    df = Phi * (H_pos / (k + H_pos)) - phi_B * B_pos - phi_H * H_pos

    # Stabilization/projection
    dB += lam * (B_pos - B)
    dH += lam * (H_pos - H)
    dT += lam * (T_pos - Tm)
    dV += lam * (V_pos - V)
    df += lam * (f_pos - f)
    dI += lam * (I_pos - I)

    DDE = jitcdde([dB, dH, dT, dV, df, dI])

    def history(_s):
        # Mites introduced at t=105 by starting integration at STARTTIME
        return [B0, H0, T0, V0, f0, I0]

    DDE.past_from_function(history)
    DDE.step_on_discontinuities()
    DDE.set_integration_parameters(
        atol=1e-8, rtol=1e-6, min_step=1e-12, max_step=DT, first_step=DT
    )

    t0 = float(DDE.t)
    start = max(STARTTIME + eps, t0)
    times = np.arange(start, STOPTIME + DT / 2, DT)

    minH = H0
    t_collapse = np.nan

    for tt in times:
        state = DDE.integrate(float(tt))
        Hval = float(state[IDX_H])

        if Hval < minH:
            minH = Hval

        if np.isnan(t_collapse) and (Hval < H_crit):
            t_collapse = float(tt)
            break

    return minH, t_collapse

# Grid search
T_vals = np.arange(0, 101, 10)
V_vals = np.arange(0, 101, 10)

minH_map = np.zeros((len(V_vals), len(T_vals)))
tc_map   = np.full((len(V_vals), len(T_vals)), np.nan)

for iV, V0 in enumerate(V_vals):
    for iT, T0 in enumerate(T_vals):
        minH, tc = build_and_run(T0=float(T0), V0=float(V0))
        minH_map[iV, iT] = minH
        tc_map[iV, iT]   = tc

collapse_map = np.isfinite(tc_map)

In [ ]:
# Print a few "near-threshold" points (first collapses in each T0 column)
print("Approximate threshold (smallest V0 that collapses) for each T0:")
for iT, T0 in enumerate(T_vals):
    idx = np.where(collapse_map[:, iT])[0]
    if len(idx) == 0:
        print(f"T0={T0:>4}: no collapse up to V0={V_vals[-1]}")
    else:
        V_thr = V_vals[idx[0]]
        print(f"T0={T0:>4}: V0 ~ {V_thr}")

# PLOTS
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

line_color = (230/255, 97/255, 0/255)

fig, ax = plt.subplots(figsize=(5, 4))

# Lighter viridis
base = plt.colormaps["viridis"]
viridis_light = LinearSegmentedColormap.from_list(
    "viridis_light", base(np.linspace(0.22, 1.00, 256))
)

# Build true cell edges over the displayed domain (prevents clipped corner artifacts)
nV, nT = minH_map.shape
T_edges = np.linspace(T_vals[0], T_vals[-1], nT + 1)
V_edges = np.linspace(V_vals[0], V_vals[-1], nV + 1)

# Centers for contour
T_centers = 0.5 * (T_edges[:-1] + T_edges[1:])
V_centers = 0.5 * (V_edges[:-1] + V_edges[1:])
TTc, VVc = np.meshgrid(T_centers, V_centers)

# Heatmap with visible grid borders
im = ax.pcolormesh(
    T_edges, V_edges, minH_map,
    shading="flat",
    cmap=viridis_light,
    edgecolors= (1, 1, 1, 0.75), #(0, 0, 0, 0.55),
    linewidth=0.6,
    antialiased=False,   # key to avoid corner-cut artifacts
    vmin=1000, vmax=10000,
    zorder=1
)

# Contour aligned to cell centers
ax.contour(
    TTc, VVc, minH_map,
    levels=[H_crit],
    colors=[line_color],
    linewidths=2.5,
    zorder=10
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Minimum hive bee population", fontsize=15)
cbar.ax.tick_params(labelsize=18, width=1.6, length=6, colors='black')
cbar.set_ticks([1000, 5000, 10000])
cbar.set_ticklabels(['1,000', '5,000', '10,000'])

ax.set_xlim(T_vals[0], T_vals[-1])
ax.set_ylim(V_vals[0], V_vals[-1])
ax.margins(x=0, y=0)

ax.set_xlabel("Initial number of virus-free mites, $T(0)$", fontsize=15)
ax.set_ylabel("Initial number of \n virus-carrying mites, $V(0)$", fontsize=15)
ax.tick_params(axis='both', labelsize=18, width=1.6, length=6, colors='black')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xticks([0, 10, 100])
ax.set_yticks([0, 10, 100])
ax.text(-0.13, 1.15, "(c)", transform=ax.transAxes, color='black', fontsize=20, va='top', ha='left')

plt.savefig("(c)_Scenario 1.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from PIL import Image, ImageOps
import numpy as np

def autocrop(im, pad=6):
    im = im.convert("RGB")
    arr = np.asarray(im).astype(np.int16)
    mask = np.any(arr < 245, axis=2)
    coords = np.argwhere(mask)
    if coords.size == 0:
        return im
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    x0 = max(0, x0 - pad); y0 = max(0, y0 - pad)
    x1 = min(im.size[0], x1 + pad); y1 = min(im.size[1], y1 + pad)
    return im.crop((x0, y0, x1, y1))

def pad_to_height(im, target_h):
    w, h = im.size
    if h == target_h:
        return im
    top = (target_h - h) // 2
    bottom = target_h - h - top
    return ImageOps.expand(im, border=(0, top, 0, bottom), fill="white")

# Load and crop 3 images
im_jan = autocrop(Image.open("(a)_Scenario 1.png"), pad=6)
im_apr = autocrop(Image.open("(b)_Scenario 1.png"), pad=6)
im_oct = autocrop(Image.open("(c)_Scenario 1.png"), pad=6)

# Match heights
target_h = max(im_jan.size[1], im_apr.size[1], im_oct.size[1])
im_jan = pad_to_height(im_jan, target_h)
im_apr = pad_to_height(im_apr, target_h)
im_oct = pad_to_height(im_oct, target_h)

# White gaps
gap = 40
total_w = im_jan.size[0] + gap + im_apr.size[0] + gap + im_oct.size[0]

# Stitch
combined = Image.new("RGB", (total_w, target_h), (255, 255, 255))
x = 0
for im in [im_jan, im_apr, im_oct]:
    combined.paste(im, (x, 0))
    x += im.size[0] + gap

combined.save("row_1.png", dpi=(600, 600))
print("Saved: row_1.png")
